In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Qwen2.5-72B-Instruct project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-72B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
#     temperature=0.0,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|████████████████████████████████████████████████| 37/37 [00:56<00:00,  1.52s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [2]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "After_Removal_High_qwen_72B_predictions.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None


# At the beginning, before the loop
progress_file = paths.PREDICTIONS / "Qwen_72B_predictions_ORIGINAL_progress.csv"

# Check if progress file exists and load it
if os.path.exists(progress_file):
    existing_results = pd.read_csv(progress_file)
    # Extract processed row indices from QA_ID (format: "Merge Q123")
    processed_indices = set()
    for qa_id in existing_results['QA_ID']:
        # Extract number from "Merge Q123" -> 123, then convert to 0-indexed (122)
        idx = int(qa_id.split('Q')[1]) - 1
        processed_indices.add(idx)
    
    results = existing_results.to_dict('records')
    print(f"Found {len(processed_indices)} already processed rows. Resuming...")
else:
    processed_indices = set()
    results = []
    print("Starting from scratch...")


# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    # Skip if already processed
    if idx in processed_indices:
        print(f"Skipping row {idx+1}/{total_rows} (already processed)...")
        continue
    
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["context"]
        question = row["question_options"]
        
        # Improved prompt with clearer instructions
        query_full = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
        )
    
        # Generate prediction using the model
        inputs = tokenizer(query_full, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode the generated response
        raw_response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Extract the answer letter using improved function
        extracted_answer = extract_answer_letter(raw_response)
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        processed_indices.add(idx)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        processed_indices.add(idx)

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "Qwen_72B_predictions_ORIGINAL.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['QA_ID', 'context', 'question_options', 'answer_df3', 'data_source_corr', 'Origin', '70B_sentence_ids', '70B_After_Removal', 'human_sentence_ids', 'Low_Irr_70B', 'Extra_Context_Minus_70B', 'gpt_direct_prediction', '72B_Sentence_Contents', '72B_Low_Irr']
Found 780 already processed rows. Resuming...
Processing 1297 rows...
Skipping row 1/1297 (already processed)...
Skipping row 2/1297 (already processed)...
Skipping row 3/1297 (already processed)...
Skipping row 4/1297 (already processed)...
Skipping row 5/1297 (already processed)...
Skipping row 6/1297 (already processed)...
Skipping row 7/1297 (already processed)...
Skipping row 8/1297 (already processed)...
Skipping row 9/1297 (already processed)...
Skipping row 10/1297 (already processed)...
Skipping row 11/1297 (already processed)...
Skipping row 12/1297 (already processed)...
Skipping row 13/1297 (already processed)...
Skipping row 14/1297 (already processed)...
Skipping row 15/1297 (already processed)...
Skip

⚠️ Could not extract answer from response for row 781:
Response: To support the hypothesis that stress from studying for exams adversely affects the immune system, l...
✅ Processed Merge Q781: Answer = None
Processing row 782/1297...
✅ Processed Merge Q782: Answer = B
Processing row 783/1297...
✅ Processed Merge Q783: Answer = A
Processing row 784/1297...
✅ Processed Merge Q784: Answer = B
Processing row 785/1297...
✅ Processed Merge Q785: Answer = D
Processing row 786/1297...
✅ Processed Merge Q786: Answer = B
Processing row 787/1297...
✅ Processed Merge Q787: Answer = D
Processing row 788/1297...
✅ Processed Merge Q788: Answer = B
Processing row 789/1297...
⚠️ Could not extract answer from response for row 789:
Response: To calculate the statistical power, we need to understand the relationship between the alpha (α) and...
✅ Processed Merge Q789: Answer = None
Processing row 790/1297...
✅ Processed Merge Q790: Answer = B
Saved progress to CSV after 790 items
Processing row 791/1297..

✅ Processed Merge Q885: Answer = D
Processing row 886/1297...
✅ Processed Merge Q886: Answer = D
Processing row 887/1297...
✅ Processed Merge Q887: Answer = A
Processing row 888/1297...
✅ Processed Merge Q888: Answer = D
Processing row 889/1297...
✅ Processed Merge Q889: Answer = A
Processing row 890/1297...
✅ Processed Merge Q890: Answer = D
Saved progress to CSV after 890 items
Processing row 891/1297...
✅ Processed Merge Q891: Answer = D
Processing row 892/1297...
✅ Processed Merge Q892: Answer = D
Processing row 893/1297...
✅ Processed Merge Q893: Answer = A
Processing row 894/1297...
✅ Processed Merge Q894: Answer = B
Processing row 895/1297...
✅ Processed Merge Q895: Answer = C
Processing row 896/1297...
✅ Processed Merge Q896: Answer = A
Processing row 897/1297...
✅ Processed Merge Q897: Answer = D
Processing row 898/1297...
✅ Processed Merge Q898: Answer = D
Processing row 899/1297...
✅ Processed Merge Q899: Answer = D
Processing row 900/1297...
✅ Processed Merge Q900: Answer =

✅ Processed Merge Q1002: Answer = A
Processing row 1003/1297...
✅ Processed Merge Q1003: Answer = D
Processing row 1004/1297...
✅ Processed Merge Q1004: Answer = A
Processing row 1005/1297...
✅ Processed Merge Q1005: Answer = D
Processing row 1006/1297...
✅ Processed Merge Q1006: Answer = B
Processing row 1007/1297...
✅ Processed Merge Q1007: Answer = B
Processing row 1008/1297...
✅ Processed Merge Q1008: Answer = D
Processing row 1009/1297...
✅ Processed Merge Q1009: Answer = B
Processing row 1010/1297...
✅ Processed Merge Q1010: Answer = C
Saved progress to CSV after 1010 items
Processing row 1011/1297...
✅ Processed Merge Q1011: Answer = D
Processing row 1012/1297...
✅ Processed Merge Q1012: Answer = A
Processing row 1013/1297...
✅ Processed Merge Q1013: Answer = D
Processing row 1014/1297...
✅ Processed Merge Q1014: Answer = D
Processing row 1015/1297...
✅ Processed Merge Q1015: Answer = A
Processing row 1016/1297...
✅ Processed Merge Q1016: Answer = C
Processing row 1017/1297...
✅

✅ Processed Merge Q1113: Answer = C
Processing row 1114/1297...
✅ Processed Merge Q1114: Answer = B
Processing row 1115/1297...
✅ Processed Merge Q1115: Answer = A
Processing row 1116/1297...
✅ Processed Merge Q1116: Answer = A
Processing row 1117/1297...
✅ Processed Merge Q1117: Answer = B
Processing row 1118/1297...
✅ Processed Merge Q1118: Answer = B
Processing row 1119/1297...
✅ Processed Merge Q1119: Answer = B
Processing row 1120/1297...
✅ Processed Merge Q1120: Answer = A
Saved progress to CSV after 1120 items
Processing row 1121/1297...
✅ Processed Merge Q1121: Answer = A
Processing row 1122/1297...
✅ Processed Merge Q1122: Answer = C
Processing row 1123/1297...
✅ Processed Merge Q1123: Answer = C
Processing row 1124/1297...
⚠️ Could not extract answer from response for row 1124:
Response: To ensure I provide the correct response, I will analyze the key points from the case summary:

- Pr...
✅ Processed Merge Q1124: Answer = None
Processing row 1125/1297...
✅ Processed Merge Q1

⚠️ Could not extract answer from response for row 1231:
Response: To ensure you receive the correct response, please provide the patient's age, sex, and a detailed de...
✅ Processed Merge Q1231: Answer = None
Processing row 1232/1297...
✅ Processed Merge Q1232: Answer = C
Processing row 1233/1297...
✅ Processed Merge Q1233: Answer = A
Processing row 1234/1297...
✅ Processed Merge Q1234: Answer = B
Processing row 1235/1297...
✅ Processed Merge Q1235: Answer = C
Processing row 1236/1297...
✅ Processed Merge Q1236: Answer = A
Processing row 1237/1297...
✅ Processed Merge Q1237: Answer = A
Processing row 1238/1297...
✅ Processed Merge Q1238: Answer = C
Processing row 1239/1297...
✅ Processed Merge Q1239: Answer = D
Processing row 1240/1297...
✅ Processed Merge Q1240: Answer = C
Saved progress to CSV after 1240 items
Processing row 1241/1297...
✅ Processed Merge Q1241: Answer = D
Processing row 1242/1297...
✅ Processed Merge Q1242: Answer = B
Processing row 1243/1297...
✅ Processed Merge Q1

In [3]:
import pandas as pd
import re
import numpy as np
from scipy import stats

# Load the model predictions
output_df = pd.read_csv(paths.PREDICTIONS / "Qwen_72B_predictions_ORIGINAL.csv")
df = pd.read_csv(paths.DATA / "After_Removal_High_qwen_72B_predictions.csv")

# Merge the two dataframes on ID_corr
merged_df = pd.merge(df, output_df[['Origin', 'Extracted_Answer']], on='Origin', how='inner')

# Compare answers
merged_df['72B_on_70B_Match'] = merged_df['answer_df3'] == merged_df['Extracted_Answer']
merged_df['72B_on_70B_Match'] = merged_df['72B_on_70B_Match'].map({True: 'TRUE', False: 'FALSE'})

# Exact match as 0/1 for statistics
merged_df['match'] = (merged_df['answer_df3'] == merged_df['Extracted_Answer']).astype(int)

# Overall accuracy statistics
accuracy = merged_df['match'].mean()
std_dev = merged_df['match'].std()
n = len(merged_df)
se = std_dev / np.sqrt(n)
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Category-wise analysis (if data_source_corr exists in df)
if 'data_source_corr' in df.columns:
    merged_df['data_source_corr'] = merged_df['data_source_corr']
    category_stats = merged_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()

    # Calculate 95% CI per category
    ci_lower, ci_upper = [], []
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100

    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()

# Summary table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.5366 (53.66%)
Standard Deviation: 0.4988
95% Confidence Interval: [0.5094, 0.5638]
95% CI (percentage): [50.94%, 56.38%]
Sample Size: 1297

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama    582       0.546392 0.498271 0.020654     0.505826     0.586957        54.639175  49.827141      50.582612      58.695738
      medbullets    207       0.652174 0.477435 0.033184     0.586750     0.717598        65.217391  47.743511      58.675004      71.759778
        medxpert    315       0.247619 0.432316 0.024358     0.199693     0.295545        24.761905  43.231606      19.969303      29.554507
            mmlu    193       0.854922 0.353095 0.025416     0.804791     0.905053        85.492228  35.309512      80.479117      90.505339


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.5366 (53.66%)